# Chronological Train / Validation / Test Split

Random splitting is inappropriate for this forecasting task because nearby rows are temporally dependent and randomization can mix later observations into earlier development data. This workflow partitions the already-engineered chronological dataset contiguously: earliest 70% for training, next 15% for validation, latest 15% for test. Counts are calculated from the actual input.

Training is used later to fit model parameters. Validation is for candidate comparison and tuning. Test is reserved and must remain untouched until a final model is selected; no predictions or model metrics are produced here. The Phase 5 lags/rolling values were computed before splitting and use only information through each row's forecast origin, so valid history may cross split boundaries. Do not regenerate them per split or discard early validation/test rows. No scaling is done; later learned preprocessing must fit on training data only.

In [1]:
from pathlib import Path
import sys
import pandas as pd

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidate_roots if (p / 'data/processed/energydata_features.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate data/processed/energydata_features.csv')
sys.path.insert(0, str(PROJECT_ROOT))
from ml.src.split_data import split_dataset, DATE_COLUMN, TARGET_COLUMN
splits, report = split_dataset()
print(f'Source rows: {report["total_rows"]}; model feature count: {report["number_of_model_features"]}')
print(f'Target: {TARGET_COLUMN}; date retained for traceability: {DATE_COLUMN}')
for name, frame in splits.items():
    print(f'{name.title()}: {len(frame)} rows ({report["split_percentages"][name]}%)')
    print(f'  {frame[DATE_COLUMN].min()} through {frame[DATE_COLUMN].max()}')
    print(f'  SHA-256: {report["split_checksums"][name]}')
print('Boundary checks:', report['boundary_checks'])

Source rows: 18726; model feature count: 42
Target: target_appliances_next; date retained for traceability: date
Train: 13108 rows (69.998932%)
  2016-01-18 17:00:00 through 2016-04-18 17:30:00
  SHA-256: b4e74536ad61bce5a75b044066f9012348a12cbd69095a0e5a801642fc7ebbe4
Validation: 2809 rows (15.000534%)
  2016-04-18 17:40:00 through 2016-05-08 05:40:00
  SHA-256: 5b93ccdb622696544eb3abb7d59b59445bf37c5ff6dfe2f8449b4b1282e41533
Test: 2809 rows (15.000534%)
  2016-05-08 05:50:00 through 2016-05-27 17:50:00
  SHA-256: 227efb785588e32e297acdbbb7595102c210b25b30492e8acef5effddc520600
Boundary checks: {'train_before_validation': True, 'validation_before_test': True, 'train_validation_timestamps_disjoint': True, 'train_test_timestamps_disjoint': True, 'validation_test_timestamps_disjoint': True}


## Feature / Target Separation

`date` remains in each saved split for traceability, but is excluded from `X`. `target_appliances_next` is separated into `y` and never included in model inputs. Current `Appliances` stays among the numeric candidate inputs. The saved test partition is not used for model development; its descriptive target summary below is only a distribution diagnostic.

In [2]:
model_features = report['model_feature_columns']
X_train = splits['train'][model_features]
y_train = splits['train'][TARGET_COLUMN]
X_validation = splits['validation'][model_features]
y_validation = splits['validation'][TARGET_COLUMN]
X_test = splits['test'][model_features]
y_test = splits['test'][TARGET_COLUMN]
assert DATE_COLUMN not in model_features and TARGET_COLUMN not in model_features
assert 'Appliances' in model_features
assert len(y_train) + len(y_validation) + len(y_test) == report['total_rows']
assert all(report['boundary_checks'].values())
print(f'date excluded from X: {DATE_COLUMN not in X_train.columns}')
print(f'target excluded from X: {TARGET_COLUMN not in X_train.columns}')
print(f'Current Appliances retained in X: {"Appliances" in X_train.columns}')
print('Descriptive target distribution only (no model evaluation):')
for name, frame in splits.items():
    print(f'\n{name.title()}')
    display(frame[TARGET_COLUMN].agg(['mean', 'median', 'std', 'min', 'max']).to_frame('target_appliances_next'))
print('No model was trained; no scaler or other learned preprocessing was fitted.')

date excluded from X: True
target excluded from X: True
Current Appliances retained in X: True
Descriptive target distribution only (no model evaluation):

Train


,target_appliances_next
mean,97.610619
median,60.000000
std,104.665764
min,10.000000
max,1070.000000



Validation


,target_appliances_next
mean,92.961908
median,60.000000
std,89.992001
min,20.000000
max,870.000000



Test


,target_appliances_next
mean,97.027412
median,60.000000
std,91.455232
min,20.000000
max,850.000000


No model was trained; no scaler or other learned preprocessing was fitted.
